# Yulara (DKASC)

## Paths

In [ ]:
from pathlib import Path

ROOT = Path("../..").resolve()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

@dataclass(frozen=True)
class YularaConfig:

    raw_path: str = str(RAW_DIR / "5-Site_DG-PV1-DB-DG-M1A.csv")
    output_path: str = str(OUT_DIR / "Yulara.csv")

    target_year: int = 2020

    latitude: float = -25.24
    longitude: float = 130.98
    altitude_m: float = 450.0
    tilt_deg: float = 15.0
    azimuth_deg: float = 0.0

    source_timezone: str = "Australia/Darwin"

    interval_center_offset_min: int = 30

    nameplate_capacity_kw: float = 1058.4

    time_col: str = "timestamp"
    power_col: str = "Active_Power"
    ghi_col: str = "Global_Horizontal_Radiation"
    temp_col: str = "Weather_Temperature_Celsius"
    wind_col: str = "Wind_Speed"

    leakage_cols: tuple = ("Current_Phase_Average_Mean", "Active_Energy_Delivered_Received")

    max_ghi: float = 1500.0
    temp_bounds: tuple = (-20.0, 60.0)
    wind_bounds: tuple = (0.0, 50.0)

    min_readings_per_hour: int = 9

    interpolate_limit_hours: int = 2
    climatology_window_days: int = 7

    @property
    def nameplate_capacity_w(self) -> float:
        return self.nameplate_capacity_kw * 1000.0

CFG = YularaConfig()

## Load raw data

In [ ]:
def load_raw(cfg: YularaConfig) -> pd.DataFrame:

    try:
        df = pd.read_csv(cfg.raw_path)
    except pd.errors.ParserError:
        df = pd.read_csv(cfg.raw_path, on_bad_lines="skip", low_memory=False)

    df[cfg.time_col] = pd.to_datetime(df[cfg.time_col], errors="coerce", format="mixed")
    df = df.dropna(subset=[cfg.time_col]).set_index(cfg.time_col).sort_index()

    dropped = [c for c in cfg.leakage_cols if c in df.columns]
    if dropped:
        df = df.drop(columns=dropped)

    return df

df_raw = load_raw(CFG)

## Physical cleaning

In [ ]:
def clean_physical(df: pd.DataFrame, cfg: YularaConfig) -> pd.DataFrame:
    df = df.copy()

    num_cols = df.select_dtypes("number").columns
    df[num_cols] = df[num_cols].mask((df[num_cols] < -999) | (df[num_cols] > 5e4))

    if cfg.ghi_col in df.columns:
        df.loc[df[cfg.ghi_col] < 0, cfg.ghi_col] = 0.0
        df.loc[df[cfg.ghi_col] > cfg.max_ghi, cfg.ghi_col] = np.nan

    if cfg.power_col in df.columns:
        df.loc[df[cfg.power_col] < 0, cfg.power_col] = 0.0
        over = df[cfg.power_col] > cfg.nameplate_capacity_kw
        df.loc[over, cfg.power_col] = np.nan

    if cfg.temp_col in df.columns:
        lo, hi = cfg.temp_bounds
        df[cfg.temp_col] = df[cfg.temp_col].where(df[cfg.temp_col].between(lo, hi))
    if cfg.wind_col in df.columns:
        lo, hi = cfg.wind_bounds
        df[cfg.wind_col] = df[cfg.wind_col].where(df[cfg.wind_col].between(lo, hi))

    return df

df_clean = clean_physical(df_raw, CFG)

## Hourly aggregation and POA (Erbs + Perez)

In [ ]:
def aggregate_hourly(df: pd.DataFrame, cfg: YularaConfig) -> pd.DataFrame:

    cols = {cfg.power_col: "Power_kW", cfg.ghi_col: "GHI",
            cfg.temp_col: "Temp_Ambient", cfg.wind_col: "Wind_Speed"}
    sub = df[[c for c in cols if c in df.columns]].rename(columns=cols)

    hourly = sub.resample("h").mean()
    counts = sub.resample("h").count()

    enough = counts.min(axis=1) >= cfg.min_readings_per_hour
    hourly = hourly[enough]

    return hourly

def add_solar_geometry(hourly: pd.DataFrame, cfg: YularaConfig) -> pd.DataFrame:
    import pvlib
    from pvlib.irradiance import get_total_irradiance, get_extra_radiation
    from pvlib.solarposition import get_solarposition

    df = hourly.copy()

    idx_local = df.index.tz_localize(cfg.source_timezone, ambiguous="NaT", nonexistent="NaT")
    valid = idx_local.notna()
    df, idx_local = df[valid], idx_local[valid]

    idx_center = idx_local + pd.Timedelta(minutes=cfg.interval_center_offset_min)

    solar_pos = get_solarposition(idx_center, cfg.latitude, cfg.longitude, altitude=cfg.altitude_m)
    dni_dhi = pvlib.irradiance.erbs(df["GHI"].values, solar_pos["zenith"].values, idx_center)
    dni_extra = get_extra_radiation(idx_center)

    poa = get_total_irradiance(
        surface_tilt=cfg.tilt_deg, surface_azimuth=cfg.azimuth_deg,
        solar_zenith=solar_pos["zenith"].values, solar_azimuth=solar_pos["azimuth"].values,
        dni=dni_dhi["dni"].values, ghi=df["GHI"].values, dhi=dni_dhi["dhi"].values,
        dni_extra=dni_extra.values, model="perez",
    )
    df["Irradiance_POA"] = np.asarray(poa["poa_global"]).clip(min=0)
    df["_elevation"] = solar_pos["elevation"].values

    df["_utc_index"] = idx_local.tz_convert("UTC")

    df.index = df.index.tz_localize(None)
    return df

df_hourly = aggregate_hourly(df_clean, CFG)
df_hourly = add_solar_geometry(df_hourly, CFG)

## Relative humidity (NASA POWER)

In [ ]:
def peak_hour(series: pd.Series, hours: np.ndarray, mask: np.ndarray) -> int:
    return int(pd.Series(series[mask].values).groupby(hours[mask]).mean().idxmax())

def fetch_nasa_aligned(df: pd.DataFrame, cfg: YularaConfig) -> pd.DataFrame:
    import requests

    idx_local = df.index
    url = (
        "https://power.larc.nasa.gov/api/temporal/hourly/point"
        "?parameters=RH2M,ALLSKY_SFC_SW_DWN&community=RE"
        f"&longitude={cfg.longitude}&latitude={cfg.latitude}"
        f"&start={idx_local.min():%Y%m%d}&end={idx_local.max():%Y%m%d}&format=JSON"
    )

    empty = pd.DataFrame({"Humidity": np.nan, "GHI_nasa": np.nan}, index=idx_local)
    try:
        res = requests.get(url, timeout=180)
        res.raise_for_status()
        p = res.json()["properties"]["parameter"]
    except Exception:
        return empty

    def to_series(d: dict) -> pd.Series:
        s = pd.Series(list(d.values()),
                      index=pd.to_datetime(list(d.keys()), format="%Y%m%d%H"))
        return s.mask(s < -900)

    rh, gh = to_series(p["RH2M"]), to_series(p["ALLSKY_SFC_SW_DWN"])

    hours = idx_local.hour.values
    day = (df["_elevation"] > 5).values
    own_peak = peak_hour(df["GHI"], hours, day)

    ref_peak = own_peak

    best_shift, best_gap = 0, 99
    for sh in range(-12, 13):
        probe = pd.Series((idx_local + pd.Timedelta(hours=sh)).map(gh).values, index=idx_local)
        if probe.notna().mean() < 0.5:
            continue
        pk = peak_hour(probe, hours, day)
        gap = min(abs(pk - ref_peak), 24 - abs(pk - ref_peak))
        if gap < best_gap:
            best_shift, best_gap = sh, gap

    shifted_idx = idx_local + pd.Timedelta(hours=best_shift)
    out = pd.DataFrame({
        "Humidity": shifted_idx.map(rh),
        "GHI_nasa": shifted_idx.map(gh),
    }, index=idx_local)

    return out

_year_mask = df_hourly.index.year == CFG.target_year

_nasa = fetch_nasa_aligned(df_hourly[_year_mask], CFG)

df_hourly["Humidity"] = np.nan
df_hourly.loc[_year_mask, "Humidity"] = _nasa["Humidity"].values

## Target year

In [ ]:
def impute_with_tracking(df: pd.DataFrame, cols: list[str], cfg: YularaConfig) -> pd.DataFrame:
    df = df.copy()
    hours, doys = df.index.hour, df.index.dayofyear

    for col in cols:
        if col not in df.columns:
            continue
        flag = df[col].isnull()
        df[f"{col}_Imputed"] = flag
        if not flag.any():
            continue

        df[col] = df[col].interpolate(method="linear", limit=cfg.interpolate_limit_hours)

        still = df[col].isnull()
        for idx in df.index[still]:
            h, d = idx.hour, idx.dayofyear
            near = (hours == h) & (np.abs(doys - d) <= cfg.climatology_window_days)
            df.loc[idx, col] = df.loc[near, col].mean()

    return df

def build_target_year(df_hourly: pd.DataFrame, cfg: YularaConfig) -> pd.DataFrame:

    full = pd.date_range(f"{cfg.target_year}-01-01 00:00:00",
                         f"{cfg.target_year}-12-31 23:00:00", freq="h")
    df = df_hourly[df_hourly.index.year == cfg.target_year].reindex(full)

    df = df.drop(columns=["_utc_index", "_elevation"], errors="ignore")
    df = impute_with_tracking(df, ["GHI", "Temp_Ambient", "Wind_Speed",
                                   "Humidity", "Power_kW", "Irradiance_POA"], cfg)

    df["Power_Actual_W"] = df["Power_kW"] * 1000.0
    df["Power_PU"] = (df["Power_Actual_W"] / cfg.nameplate_capacity_w).clip(0, 1)

    hour, month = df.index.hour, df.index.month
    df["HOUR_SIN"] = np.sin(2 * np.pi * hour / 24.0)
    df["HOUR_COS"] = np.cos(2 * np.pi * hour / 24.0)
    df["MONTH_SIN"] = np.sin(2 * np.pi * month / 12.0)
    df["MONTH_COS"] = np.cos(2 * np.pi * month / 12.0)

    df["Latitude"], df["Longitude"], df["Tilt"] = cfg.latitude, cfg.longitude, cfg.tilt_deg
    df["utctime"] = df.index

    schema = ["utctime", "Power_Actual_W", "Power_PU", "Irradiance_POA", "GHI",
              "Temp_Ambient", "Wind_Speed", "Humidity", "HOUR_SIN", "HOUR_COS",
              "MONTH_SIN", "MONTH_COS", "Latitude", "Longitude", "Tilt"]
    flags = [c for c in df.columns if c.endswith("_Imputed")]

    out = df[schema + flags].reset_index(drop=True)
    return out

df_final = build_target_year(df_hourly, CFG)

## Save

In [ ]:
os.makedirs(os.path.dirname(CFG.output_path), exist_ok=True)
df_final.to_csv(CFG.output_path, index=False)